In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('combined.csv', low_memory=False)

In [3]:
df.columns

Index(['pickup_datetime', 'dropoff_datetime', 'ratecodeid', 'pulocationid',
       'dolocationid', 'passenger_count', 'trip_distance', 'fare_amount',
       'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
       'improvement_surcharge', 'total_amount', 'payment_type', 'trip_type',
       'congestion_surcharge', 'is_green_ride', 'airport_fee'],
      dtype='object')

In [4]:
df.head()

,pickup_datetime,dropoff_datetime,ratecodeid,pulocationid,dolocationid,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,is_green_ride,airport_fee
0,09/01/2022 12:45:23 AM,09/01/2022 12:52:39 AM,1.0,74,42,1.0,1.51,7.5,0.5,0.5,1.32,0.00,0.3,10.12,1.0,1.0,0.00,True,0.0
1,09/01/2022 12:40:58 AM,09/01/2022 01:11:42 AM,5.0,93,79,3.0,14.51,65,0.0,0.0,10.00,6.55,0.3,84.6,1.0,2.0,2.75,True,0.0
2,09/01/2022 12:03:27 AM,09/01/2022 12:08:31 AM,1.0,134,134,2.0,1.01,5.5,0.5,0.5,0.00,0.00,0.3,6.8,2.0,1.0,0.00,True,0.0
3,09/01/2022 12:31:17 AM,09/01/2022 12:35:20 AM,1.0,7,179,1.0,0.59,4.5,0.5,0.5,1.74,0.00,0.3,7.54,1.0,1.0,0.00,True,0.0
4,08/31/2022 11:58:32 PM,09/01/2022 12:15:52 AM,5.0,93,233,1.0,10.6,70,0.0,0.0,19.90,6.55,0.3,99.5,1.0,2.0,2.75,True,0.0


In [ ]:
# Convert to datetime objects
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], format='mixed')
df['dropoff_datetime'] = pd.to_datetime(df['dropoff_datetime'], format='mixed')

In [ ]:
# calculate duration in minutes
df['trip_duration'] = (df['dropoff_datetime'] - df['pickup_datetime']).dt.total_seconds() / 60
df['day_of_week'] = df['pickup_datetime'].dt.dayofweek  # Monday=0, Sunday=6

In [ ]:
df.columns

In [ ]:
df = df.sort_values('pickup_datetime').reset_index(drop=True)

In [ ]:
df = df[(df['pickup_datetime'].dt.year >= 2022) & (df['pickup_datetime'].dt.year <= 2023)]

In [ ]:
weather = pd.read_csv("weather.csv")

In [ ]:
weather.columns

In [ ]:
weather.drop(columns=[
    "name", "tempmax", "tempmin", "severerisk", "feelslikemax", 
    "feelslike", "feelslikemin", "stations", "icon", "description", 
    "sunrise", "sunset", "conditions", "solarenergy", "solarradiation",
    "winddir", "windgust", "sealevelpressure"
], inplace=True, errors="ignore")

In [ ]:
weather.head()

In [ ]:
weather["date"] = pd.to_datetime(weather["datetime"])
df['pickup_date'] = df['pickup_datetime'].dt.date
weather['date'] = weather['date'].dt.date

In [ ]:
merged_df = df.merge(weather, left_on='pickup_date', right_on='date', how='left')
merged_df.head()

In [ ]:
merged_df.columns

In [ ]:
merged_df.drop(columns=["datetime", "pickup_date", "date", "dew", "precipprob", "precipcover", "uvindex", "day_of_week", "moonphase"], inplace=True, errors="ignore")

In [ ]:
print(len(merged_df))
print(len(df))

In [ ]:
merged_df.head()

In [ ]:
merged_df.columns

In [ ]:
merged_df['pickup_hour'] = merged_df['pickup_datetime'].dt.hour
merged_df['pickup_month'] = merged_df['pickup_datetime'].dt.month
merged_df['pickup_day_of_week'] = merged_df['pickup_datetime'].dt.dayofweek # 0=Monday, 6=Sunday

In [ ]:
len(merged_df)

In [ ]:
merged_df['trip_distance'] = merged_df['trip_distance'].str.replace(',', '').astype(float)

In [ ]:
# Remove non-sensical trips
merged_df = merged_df[(merged_df['trip_distance'] > 0) & (merged_df['trip_distance'] < 100)]
merged_df = merged_df[(merged_df['trip_duration'] > 1) & (merged_df['trip_duration'] < 240)] # 1 min to 4 hours

len(merged_df)

In [ ]:
merged_df.to_parquet("combined.parquet", index=False)